# GTHDM investigation

Scratch notebook to inspect the pymarty GTHDM port: particles, Lagrangian terms, and LaTeX output.

Setup: run the kernel with the pymarty build on `PYTHONPATH`, for example
`PYTHONPATH=/tmp/pymarty-build/python`, and a kernel that has `ipykernel` installed.
Building `GTHDM()` takes a few minutes; rerun the construction cell only after code changes.

The Lagrangian is grouped by `kind`: `"kinetic"`, `"mass"` and `"interaction"`.

In [10]:
from IPython.display import Math, display

from pymarty.modeling.models import GTHDM
from pymarty.modeling.models._symbols import constant as shared_constant

In [11]:
model = GTHDM()
print(f"particles: {len(model.particles)}")

Linear : -1/2*v1^3*lambda_1*rho_1(X)
Linear : v1*m_12^2*tan(beta)*rho_1(X)
Linear : -1/2*v1^3*lambda_3*tan(beta)^2*rho_1(X)
Linear : -1/2*v1^3*lambda_4*tan(beta)^2*rho_1(X)
Linear : -1/2*v1^3*lambda_5*tan(beta)^2*rho_1(X)
Linear : v1*(1/2*v1^2*lambda_1 + -m_12^2*tan(beta) + 1/2*v1^2*lambda_3*tan(beta)^2 + 1/2*v1^2*lambda_4*tan(beta)^2 + 1/2*v1^2*lambda_5*tan(beta)^2)*rho_1(X)
Linear : v1*m_12^2*rho_2(X)
Linear : -1/2*v1^3*lambda_3*tan(beta)*rho_2(X)
Linear : -1/2*v1^3*lambda_4*tan(beta)*rho_2(X)
Linear : -1/2*v1^3*lambda_5*tan(beta)*rho_2(X)
Linear : -1/2*v1^3*lambda_2*tan(beta)^3*rho_2(X)
Linear : v1*(-m_12^2 + 1/2*v1^2*lambda_3*tan(beta) + 1/2*v1^2*lambda_4*tan(beta) + 1/2*v1^2*lambda_5*tan(beta) + 1/2*v1^2*lambda_2*tan(beta)^3)*rho_2(X)



particles: 45


In [12]:
print(sorted(p.name for p in model.particles))

['A', 'A0', 'G', 'G0', 'Gp', 'H0', 'Hp', 'W', 'Z', 'b', 'b_L', 'b_R', 'c', 'c_A', 'c_G', 'c_L', 'c_R', 'c_Wm', 'c_Wp', 'c_Z', 'd', 'd_L', 'd_R', 'e', 'e_L', 'e_R', 'h', 'mu', 'mu_L', 'mu_R', 'nu_e', 'nu_mu', 'nu_tau', 's', 's_L', 's_R', 't', 't_L', 't_R', 'tau', 'tau_L', 'tau_R', 'u', 'u_L', 'u_R']


In [13]:
terms = {
    kind: model.lagrangian_terms(kind) for kind in ("kinetic", "mass", "interaction")
}
for kind, items in terms.items():
    print(f"{kind}: {len(items)} terms")

kinetic: 35 terms
mass: 18 terms
interaction: 443 terms


In [20]:
for index, mass_term in enumerate(model.lagrangian_terms("mass")):
    print(f"Mass term {index}:", end="")
    display(Math(mass_term.to_latex()))

Mass term 0:

<IPython.core.display.Math object>

Mass term 1:

<IPython.core.display.Math object>

Mass term 2:

<IPython.core.display.Math object>

... (26 outputs left out)

Mass term 16:

<IPython.core.display.Math object>

Mass term 17:

<IPython.core.display.Math object>

In [15]:
def top_level_terms(expr):
    """Split an expression string on ' + ' outside any brackets."""
    parts, depth, current, i = [], 0, "", 0
    text = str(expr)
    while i < len(text):
        ch = text[i]
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth -= 1
        if depth == 0 and text.startswith(" + ", i):
            parts.append(current)
            current, i = "", i + 3
            continue
        current += ch
        i += 1
    parts.append(current)
    return parts


def textual_cancellations(kind):
    """Yield (term index, component) for components whose negative also appears."""
    for idx, term in enumerate(terms[kind]):
        comps = top_level_terms(term)
        for comp in comps:
            neg = comp.startswith("-")
            key = comp[1:] if neg else comp
            if neg and key in comps and key != comp:
                yield idx, key


found = list(textual_cancellations("mass")) + list(textual_cancellations("interaction"))
print(f"textually cancelling pairs: {len(found)}")
for idx, key in found[:5]:
    print(idx, key[:150])

textually cancelling pairs: 0


In [16]:
# Inspect the summands of one term and check whether two of them cancel symbolically.
# The index 5 is an arbitrary example; set it to a term you want to examine.
expr = terms["mass"][5]
for arg in expr.args:
    print(str(arg)[:200])

pair_sum = expr.args[1] + expr.args[2]
print("pair sum expanded:", pair_sum.expand())

Fraction: 1/2
M_Z^2
Z_%\sigma_2371(X)
Z_+%\sigma_2371(X)
pair sum expanded: M_Z^2 + Z_%\sigma_2371(X)


In [17]:
# Compare the shared symbol used by the builders with a fresh pymarty constant.
from pymarty.symbolic import constant as fresh_constant

a = shared_constant("beta;\\beta")
b = shared_constant("beta;\\beta")
c = fresh_constant("beta;\\beta")
print("shared minus shared:", (a - b).expand())
print("shared minus fresh: ", (a - c).expand())

shared minus shared: 0
shared minus fresh:  beta + -beta
